In [2]:
!nvidia-smi

Mon Oct 26 10:29:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import files
uploaded = files.upload()

Saving hybrid_mpi_openmp_cuda_railway.zip to hybrid_mpi_openmp_cuda_railway.zip


In [4]:
import os
import zipfile

zip_file = next(
    name for name in uploaded if name.endswith(".zip")
)

workdir = "/content/hybrid_railway"
os.makedirs(workdir, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as z:
    z.extractall(workdir)

print("Extracted files:")
print(os.listdir(workdir))

Extracted files:
['cuda_conflicts.h', 'hybrid_railway.cpp', 'events.csv', 'cuda_conflicts.cu', 'trains.csv', 'README.txt']


## Revised hybrid railway allocator
The program compares Greedy, Welsh-Powell, and DSATUR, then evaluates dynamic DSATUR repair after simulated events. It reports feasibility, accommodation rate, conflict violations, platform utilization, reassignment counts, and stage timings. The provided data are simulated.

In [ ]:
%cd /content/hybrid_railway
!nvcc -O2 -c cuda_conflicts.cu -o cuda_conflicts.o
!mpicxx -O2 -std=c++17 -fopenmp -c hybrid_railway.cpp -o hybrid_railway.o
!mpicxx -fopenmp hybrid_railway.o cuda_conflicts.o -L/usr/local/cuda/lib64 -lcudart -o hybrid_railway
!OMP_NUM_THREADS=4 mpirun --allow-run-as-root --oversubscribe -np 2 ./hybrid_railway trains.csv events.csv